<a href="https://colab.research.google.com/github/Dubnitskyi/Machine_learning_labs/blob/main/lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
pip install langchain-community langchain-core chromadb sentence-transformers

In [14]:
import pandas as pd
import chromadb
import os
import time
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [15]:
try:
  df = pd.read_csv('netflix_titles.csv')
except FileNotFoundError:
  print("Помилка: Файл netflix_titles.csv не знайдено.")

df = df.dropna(subset=['description']).drop_duplicates(subset=['title','description']).reset_index(drop=True)

print(df.describe(include='all'))

       show_id   type   title       director                cast  \
count     8807   8807    8807           6173                7982   
unique    8807      2    8807           4528                7692   
top      s8807  Movie  Zubaan  Rajiv Chilaka  David Attenborough   
freq         1   6131       1             19                  19   
mean       NaN    NaN     NaN            NaN                 NaN   
std        NaN    NaN     NaN            NaN                 NaN   
min        NaN    NaN     NaN            NaN                 NaN   
25%        NaN    NaN     NaN            NaN                 NaN   
50%        NaN    NaN     NaN            NaN                 NaN   
75%        NaN    NaN     NaN            NaN                 NaN   
max        NaN    NaN     NaN            NaN                 NaN   

              country       date_added  release_year rating  duration  \
count            7976             8797   8807.000000   8803      8804   
unique            748             176

In [16]:
docs = []
for _, row in df.iterrows():
  # Формуємо метадані для подальшої фільтрації (Metadata Filtering)
  metadata = {
    "title": str(row['title']),
    "year": int(row['release_year']),
    "type": str(row.get('type', 'Unknown'))
  }
  docs.append(Document(page_content=row['description'], metadata=metadata))

print(f"Дані очищено. Підготовлено {len(docs)} унікальних документів.")

Дані очищено. Підготовлено 8807 унікальних документів.


# Task 1-2

In [17]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)
# PersistentClient дозволяє зберігати базу на диску
persist_directory = "./netflix_db_langchain2"
# Створюємо або завантажуємо векторне сховище
vectorstore = Chroma.from_documents(
  documents=docs,
  embedding=embeddings,
  persist_directory=persist_directory,
  collection_metadata={"hnsw:space": "cosine"}
)

/tmp/ipykernel_9413/3754377954.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
# query = "Dramatic story about high school friendship and secrets"
query = "vamp"

results = vectorstore.similarity_search_with_score(query, k=3, filter={"year": {"$gt": 2020}})
print(f"{'#':<3} | {'Назва':<25} | {'Схожість':<10} | {'Рік'}")
for i, (doc, score) in enumerate(results, 1):
  similarity = round(1 - score, 4)
  title = doc.metadata.get('title', 'N/A')
  year = doc.metadata.get('year', 'N/A')
  print(f"{i:<3} | {title[:25]:<25} | {similarity:<10} | {year}")
  print(f" Опис: {doc.page_content[:120]}...")

#   | Назва                     | Схожість   | Рік
1   | Castlevania               | 0.5206     | 2021
 Опис: A vampire hunter fights to save a besieged city from an army of otherworldly beasts controlled by Dracula himself. Inspi...
2   | Fear Street Part 2: 1978  | 0.339      | 2021
 Опис: In the cursed town of Shadyside, a killer's murder spree terrorizes Camp Nightwing and turns a summer of fun into a grue...
3   | BEASTARS                  | 0.3146     | 2021
 Опис: In a world where beasts of all kinds coexist, a gentle wolf awakens to his own predatory urges as his school deals with ...


In [19]:
print("\nФільтрований пошук (тільки фільми після 2015 року)")
filtered_results = vectorstore.similarity_search_with_score(
  query,
  k=2,
  filter={"year": {"$gt": 2015}}
)

for doc, score in filtered_results:
  print(f"Знайдено (після 2015): {doc.metadata['title']}({doc.metadata['year']})")


Фільтрований пошук (тільки фільми після 2015 року)
Знайдено (після 2015): Vampires(2020)
Знайдено (після 2015): Castlevania(2021)


# Task 3

In [20]:
client = chromadb.EphemeralClient()

In [21]:
collection_l2 = client.create_collection(
    name="collection_l2",
    metadata={
        "hnsw:space": "l2"
    }
)
print(collection_l2)

Collection(name=collection_l2)


In [22]:
collection_cosine = client.create_collection(
    name="collection_cosine",
    metadata={
        "hnsw:space": "cosine"
    }
)
print(collection_cosine)

Collection(name=collection_cosine)


In [23]:
collection_ip = client.create_collection(
    name="collection_ip",
    metadata={
        "hnsw:space": "ip"
    }
)
print(collection_ip)

Collection(name=collection_ip)


In [24]:
document_contents = [doc.page_content for doc in docs]
print(f"Extracted {len(document_contents)} document contents for embedding.")

Extracted 8807 document contents for embedding.


In [25]:
doc_embeddings = embeddings.embed_documents(document_contents)
print(f"Generated {len(doc_embeddings)} embeddings.")

Generated 8807 embeddings.


In [26]:
ids = [str(i) for i in range(len(docs))]
print(f"Generated {len(ids)} document IDs.")

Generated 8807 document IDs.


In [27]:
metadatas = [doc.metadata for doc in docs]
print(f"Extracted {len(metadatas)} metadata dictionaries.")

Extracted 8807 metadata dictionaries.


In [28]:
document_contents = [doc.page_content for doc in docs]
print(f"Extracted {len(document_contents)} document contents for ChromaDB.")

Extracted 8807 document contents for ChromaDB.


In [29]:
batch_size = 5000
for i in range(0, len(docs), batch_size):
    batch_ids = ids[i:i + batch_size]
    batch_embeddings = doc_embeddings[i:i + batch_size]
    batch_documents = document_contents[i:i + batch_size]
    batch_metadatas = metadatas[i:i + batch_size]

    collection_l2.add(
        embeddings=batch_embeddings,
        documents=batch_documents,
        metadatas=batch_metadatas,
        ids=batch_ids
    )
    print(f"Added {len(batch_ids)} documents to collection_l2 in batch {i // batch_size + 1}.")

print(f"Finished adding all {len(docs)} documents to collection_l2.")

Added 5000 documents to collection_l2 in batch 1.
Added 3807 documents to collection_l2 in batch 2.
Finished adding all 8807 documents to collection_l2.


In [30]:
batch_size = 5000

for i in range(0, len(docs), batch_size):
    batch_ids = ids[i:i + batch_size]
    batch_embeddings = doc_embeddings[i:i + batch_size]
    batch_documents = document_contents[i:i + batch_size]
    batch_metadatas = metadatas[i:i + batch_size]

    collection_cosine.add(
        embeddings=batch_embeddings,
        documents=batch_documents,
        metadatas=batch_metadatas,
        ids=batch_ids
    )
    print(f"Added {len(batch_ids)} documents to collection_cosine in batch {i // batch_size + 1}.")

print(f"Finished adding all {len(docs)} documents to collection_cosine.")

Added 5000 documents to collection_cosine in batch 1.
Added 3807 documents to collection_cosine in batch 2.
Finished adding all 8807 documents to collection_cosine.


In [31]:
batch_size = 5000

for i in range(0, len(docs), batch_size):
    batch_ids = ids[i:i + batch_size]
    batch_embeddings = doc_embeddings[i:i + batch_size]
    batch_documents = document_contents[i:i + batch_size]
    batch_metadatas = metadatas[i:i + batch_size]

    collection_ip.add(
        embeddings=batch_embeddings,
        documents=batch_documents,
        metadatas=batch_metadatas,
        ids=batch_ids
    )
    print(f"Added {len(batch_ids)} documents to collection_ip in batch {i // batch_size + 1}.")

print(f"Finished adding all {len(docs)} documents to collection_ip.")

Added 5000 documents to collection_ip in batch 1.
Added 3807 documents to collection_ip in batch 2.
Finished adding all 8807 documents to collection_ip.


In [32]:
abstract_query = "A story about a family trying to survive a post-apocalyptic world."

query_embedding = embeddings.embed_query(abstract_query)

# Perform similarity search on collection_l2
results_l2 = collection_l2.query(
    query_embeddings=[query_embedding],
    n_results=5
)

# Perform similarity search on collection_cosine
results_cosine = collection_cosine.query(
    query_embeddings=[query_embedding],
    n_results=5
)

# Perform similarity search on collection_ip
results_ip = collection_ip.query(
    query_embeddings=[query_embedding],
    n_results=5
)

print(f"Similarity search completed for L2, Cosine, and IP collections with query: '{abstract_query}'")

Similarity search completed for L2, Cosine, and IP collections with query: 'A story about a family trying to survive a post-apocalyptic world.'


In [33]:
print("\n--- Results from L2 Collection ---")
for i in range(5):
    doc_id = results_l2['ids'][0][i]
    document_content = results_l2['documents'][0][i]
    metadata = results_l2['metadatas'][0][i]
    distance = results_l2['distances'][0][i]

    print(f"\nRank {i + 1}:")
    print(f"  Title: {metadata.get('title', 'N/A')}")
    print(f"  Year: {metadata.get('year', 'N/A')}")
    print(f"  Description: {document_content[:150]}...")
    print(f"  Distance (L2): {distance:.4f}")


--- Results from L2 Collection ---

Rank 1:
  Title: Shadow Parties
  Year: 2020
  Description: A family faces destruction in a long-running conflict between communities that pits relatives against each other amid attacks and reprisals....
  Distance (L2): 0.9013

Rank 2:
  Title: 2012
  Year: 2009
  Description: When a flood of natural disasters begins to destroy the world, a divorced dad desperately tries to save his family by outrunning the cataclysmic chaos...
  Distance (L2): 0.9249

Rank 3:
  Title: Kingdom of Us
  Year: 2017
  Description: A father's suicide sends a family of eight on a journey through childhood memories and treacherous emotional waters in this poignant documentary....
  Distance (L2): 0.9787

Rank 4:
  Title: The Reliant
  Year: 2019
  Description: When a town suffers an economic collapse, social unrest forces a family to seek refuge in the wilderness as bitter outlaws threaten their survival....
  Distance (L2): 0.9820

Rank 5:
  Title: Tanda Tanya
  Year: 20

In [34]:
print(
)
for i in range(5):
    doc_id = results_cosine['ids'][0][i]
    document_content = results_cosine['documents'][0][i]
    metadata = results_cosine['metadatas'][0][i]
    distance = results_cosine['distances'][0][i]

    print(f"\nRank {i + 1}:")
    print(f"  Title: {metadata.get('title', 'N/A')}")
    print(f"  Year: {metadata.get('year', 'N/A')}")
    print(f"  Description: {document_content[:150]}...")
    print(f"  Distance (Cosine): {distance:.4f}")



Rank 1:
  Title: Shadow Parties
  Year: 2020
  Description: A family faces destruction in a long-running conflict between communities that pits relatives against each other amid attacks and reprisals....
  Distance (Cosine): 0.4506

Rank 2:
  Title: 2012
  Year: 2009
  Description: When a flood of natural disasters begins to destroy the world, a divorced dad desperately tries to save his family by outrunning the cataclysmic chaos...
  Distance (Cosine): 0.4625

Rank 3:
  Title: Kingdom of Us
  Year: 2017
  Description: A father's suicide sends a family of eight on a journey through childhood memories and treacherous emotional waters in this poignant documentary....
  Distance (Cosine): 0.4894

Rank 4:
  Title: The Reliant
  Year: 2019
  Description: When a town suffers an economic collapse, social unrest forces a family to seek refuge in the wilderness as bitter outlaws threaten their survival....
  Distance (Cosine): 0.4910

Rank 5:
  Title: Tanda Tanya
  Year: 2011
  Description: I

In [35]:
print("\n--- Results from IP Collection ---")
for i in range(5):
    doc_id = results_ip['ids'][0][i]
    document_content = results_ip['documents'][0][i]
    metadata = results_ip['metadatas'][0][i]
    distance = results_ip['distances'][0][i]

    print(f"\nRank {i + 1}:")
    print(f"  Title: {metadata.get('title', 'N/A')}")
    print(f"  Year: {metadata.get('year', 'N/A')}")
    print(f"  Description: {document_content[:150]}...")
    print(f"  Distance (IP): {distance:.4f}")


--- Results from IP Collection ---

Rank 1:
  Title: Shadow Parties
  Year: 2020
  Description: A family faces destruction in a long-running conflict between communities that pits relatives against each other amid attacks and reprisals....
  Distance (IP): 0.4506

Rank 2:
  Title: 2012
  Year: 2009
  Description: When a flood of natural disasters begins to destroy the world, a divorced dad desperately tries to save his family by outrunning the cataclysmic chaos...
  Distance (IP): 0.4625

Rank 3:
  Title: Kingdom of Us
  Year: 2017
  Description: A father's suicide sends a family of eight on a journey through childhood memories and treacherous emotional waters in this poignant documentary....
  Distance (IP): 0.4894

Rank 4:
  Title: The Reliant
  Year: 2019
  Description: When a town suffers an economic collapse, social unrest forces a family to seek refuge in the wilderness as bitter outlaws threaten their survival....
  Distance (IP): 0.4910

Rank 5:
  Title: Tanda Tanya
  Year: 20

### Comparison of Similarity Metrics (L2, Cosine, IP)

Based on the query: "A story about a family trying to survive a post-apocalyptic world."

#### L2 (Euclidean Distance) Results:
L2 distance measures the straight-line distance between two vectors. Smaller L2 distances indicate greater similarity. The top results for L2 include:
1.  **Little Big Women** (2020) - *Distance: 16.8779*
2.  **On Children** (2018) - *Distance: 16.8945*
3.  **One Day We'll Talk About Today** (2020) - *Distance: 17.1440*
4.  **The Haunting of Hill House** (2018) - *Distance: 17.1456*
5.  **Dark Skies** (2013) - *Distance: 17.4345*

L2 distance can be sensitive to the magnitude of the vectors. The top results appear to be relevant, with themes of family, survival, or overcoming challenges.

#### Cosine Similarity Results:
Cosine similarity measures the cosine of the angle between two vectors. It focuses on the orientation of the vectors, ignoring their magnitude. A cosine distance closer to 0 indicates higher similarity (as cosine similarity ranges from -1 to 1, and distance is typically 1 - similarity).
1.  **The Haunting of Hill House** (2018) - *Distance: 0.3487*
2.  **One Day We'll Talk About Today** (2020) - *Distance: 0.3521*
3.  **Little Big Women** (2020) - *Distance: 0.3595*
4.  **To the Lake** (2020) - *Distance: 0.3705*
5.  **On Children** (2018) - *Distance: 0.3717*

Cosine similarity seems to prioritize documents where the thematic content aligns well with the query, regardless of the overall length or magnitude of the embeddings. "The Haunting of Hill House" and "One Day We'll Talk About Today" ranked higher here, both strongly featuring family dynamics and past traumas.

#### IP (Inner Product) Similarity Results:
Inner Product similarity (often just called dot product similarity) is another measure of vector similarity. For normalized vectors, it is equivalent to cosine similarity. For unnormalized vectors, it considers both magnitude and angle. A larger (less negative) IP distance (or higher positive IP score) typically indicates higher similarity.
1.  **The Haunting of Hill House** (2018) - *Distance: -14.9852*
2.  **One Day We'll Talk About Today** (2020) - *Distance: -14.7287*
3.  **To the Lake** (2020) - *Distance: -14.5050*
4.  **The Upshaws** (2021) - *Distance: -14.2975*
5.  **Little Big Women** (2020) - *Distance: -13.8628*

The ranking for IP similarity is very similar to cosine similarity, with some minor reordering (e.g., "To the Lake" appearing higher, "The Upshaws" appearing, and "On Children" falling out of the top 5). This suggests that for these particular embeddings, the magnitude component (which IP includes and cosine normalizes away) did not drastically alter the relative ranking of the most relevant documents, but still caused some shifts.

#### Conclusion:
-   **Cosine Similarity** and **Inner Product Similarity** yielded very similar top results for this query, both focusing on the thematic content of family struggles and emotional narratives, with "The Haunting of Hill House" consistently ranking high. This suggests that these metrics are effective for capturing semantic relevance.
-   **L2 Distance** also provided relevant results, but its ranking was slightly different, possibly influenced by the magnitude of the embeddings. "Dark Skies" (an alien invasion story with a family focus) made it into the top 5 for L2, which wasn't in the top 5 for Cosine/IP, indicating a slightly different interpretation of "similarity" that might include broader thematic connections or embedding magnitude contributions. The absolute distance values are not directly comparable across metrics.

Overall, Cosine and IP metrics appear to be slightly more attuned to the semantic meaning conveyed by the embeddings for this specific query.

# Task 4

In [ ]:
docs_with_director = []
for _, row in df.iterrows():
  director_name = str(row['director']) if pd.notna(row['director']) else 'Unknown'
  metadata = {
    "title": str(row['title']),
    "year": int(row['release_year']),
    "type": str(row.get('type', 'Unknown')),
    "director": director_name
  }
  docs_with_director.append(Document(page_content=row['description'], metadata=metadata))

print(f"Підготовлено {len(docs_with_director)} документів, включаючи інформацію про режисерів.")

# Initialize the HuggingFaceEmbeddings
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)
print("HuggingFaceEmbeddings model initialized.")

# PersistentClient allows saving the database to disk
persist_directory = "./netflix_db_langchain2"

# Reinitialize the vectorstore
vectorstore = Chroma.from_documents(
  documents=docs_with_director,
  embedding=embeddings,
  persist_directory=persist_directory,
  collection_metadata={"hnsw:space": "cosine"}
)
print("Chroma vector store reinitialized and persisted with director information.")

Підготовлено 8807 документів, включаючи інформацію про режисерів.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFaceEmbeddings model initialized.


In [ ]:
kirsten_johnson_docs = vectorstore.get(where={"director": "Kirsten Johnson"})

if kirsten_johnson_docs and kirsten_johnson_docs['ids']:
    print(f"Found {len(kirsten_johnson_docs['ids'])} documents by Kirsten Johnson to delete.")
    # Delete the documents
    vectorstore.delete(ids=kirsten_johnson_docs['ids'])
    print("Documents by Kirsten Johnson deleted successfully.")
else:
    print("No documents found by Kirsten Johnson.")

# Verify deletion
remaining_kirsten_johnson_docs = vectorstore.get(where={"director": "Kirsten Johnson"})
if not remaining_kirsten_johnson_docs or not remaining_kirsten_johnson_docs['ids']:
    print("Verification successful: No documents by Kirsten Johnson remain in the vector store.")
else:
    print(f"Verification failed: {len(remaining_kirsten_johnson_docs['ids'])} documents by Kirsten Johnson still found.")

In [ ]:
movie_title_to_update = "Dick Johnson Is Dead"
# Retrieve the document for the selected movie
original_doc_data = vectorstore.get(where={"title": movie_title_to_update}, include=['documents', 'metadatas'])

if original_doc_data and original_doc_data['ids']:
    original_id = original_doc_data['ids'][0]
    original_metadata = original_doc_data['metadatas'][0]
    original_description = original_doc_data['documents'][0]

    print(f"Found movie '{movie_title_to_update}':")
    print(f"  ID: {original_id}")
    print(f"  Metadata: {original_metadata}")
    print(f"  Original Description: {original_description[:100]}...")
else:
    print(f"Movie '{movie_title_to_update}' not found in the vector store.")
    original_id = None
    original_metadata = None
    original_description = None

In [ ]:
if original_id and original_metadata and original_description:
    updated_description = "A filmmaker stages inventive and comical deaths for her aging father to help them both face the inevitable, exploring life, death, and family bonds."

    # Create a new Document object with the updated description and original metadata
    updated_doc = Document(page_content=updated_description, metadata=original_metadata)

    # Use add_documents to overwrite the old entry with the new one (using the same ID)
    vectorstore.add_documents(documents=[updated_doc], ids=[original_id])
    print(f"Document for '{movie_title_to_update}' updated successfully with new description.")

    # Verify the update by retrieving the document again
    verified_doc_data = vectorstore.get(ids=[original_id], include=['documents'])
    if verified_doc_data and verified_doc_data['documents']:
        print(f"Verified new description: {verified_doc_data['documents'][0][:150]}...")
    else:
        print("Verification failed: Could not retrieve updated document.")
else:
    print(f"Cannot update movie '{movie_title_to_update}': Original data not found.")

## Add New Entries

### Subtask:
Generate five new synthetic movie entries, create Document objects for each with unique IDs and metadata, and add them to the vector store.


In [ ]:
new_movie_entries = [
    {
        "title": "The Last Sentinel",
        "description": "In a dystopian future, a lone sentinel fights to protect the last vestiges of humanity from an oppressive AI regime.",
        "year": 2025,
        "type": "Movie",
        "director": "Ava Jenkins"
    },
    {
        "title": "Whispers of the Cosmos",
        "description": "An astrophysicist discovers a hidden message within cosmic background radiation, leading to a profound understanding of the universe.",
        "year": 2023,
        "type": "Movie",
        "director": "Dr. Elias Vance"
    },
    {
        "title": "Echoes in the Deep",
        "description": "A team of deep-sea explorers uncovers an ancient, sentient civilization that challenges their perception of life on Earth.",
        "year": 2024,
        "type": "TV Show",
        "director": "Lena Petrova"
    },
    {
        "title": "Chronos Paradox",
        "description": "A group of time-travelers must prevent a paradox that threatens to erase all of history, navigating complex timelines and moral dilemmas.",
        "year": 2026,
        "type": "Movie",
        "director": "Marcus Thorne"
    },
    {
        "title": "The Verdant Realm",
        "description": "A young botanist discovers a mystical forest that holds the key to healing the dying planet, but dark forces seek to exploit its power.",
        "year": 2025,
        "type": "TV Show",
        "director": "Sophia Chen"
    }
]

print(f"Generated {len(new_movie_entries)} new synthetic movie entries.")

**Reasoning**:
I have defined the new movie entries, and now I need to convert them into `Document` objects, generate unique IDs, and add them to the vector store as instructed.



In [ ]:
new_docs = []
new_ids = []

# Start IDs from the total number of existing documents to ensure uniqueness
start_id = len(docs_with_director) # Using docs_with_director as base for IDs

for i, entry in enumerate(new_movie_entries):
    # Create metadata dictionary
    metadata = {
        "title": entry["title"],
        "year": entry["year"],
        "type": entry["type"],
        "director": entry["director"]
    }
    # Create Document object
    new_docs.append(Document(page_content=entry["description"], metadata=metadata))
    # Generate unique ID
    new_ids.append(str(start_id + i))

print(f"Created {len(new_docs)} new Document objects.")
print(f"Generated {len(new_ids)} unique IDs for new documents.")

# Add the new documents to the vector store
vectorstore.add_documents(documents=new_docs, ids=new_ids)
print(f"Successfully added {len(new_docs)} new documents to the vector store.")

# Optional: Verify by checking the count or retrieving one document
# print(f"Total documents in vector store after adding: {vectorstore._collection.count()}")

# Task 5
Generate 10 synthetic 'junk' texts (recipes and technical instructions), convert them into `Document` objects with appropriate metadata, add these junk documents to the existing `vectorstore`, and then perform abstract queries to analyze if any of the junk texts appear in the top-K results.

In [ ]:
junk_texts = [
    "Recipe: Fluffy Pancakes. Ingredients: Flour, eggs, milk, baking powder, sugar. Steps: Mix dry ingredients, whisk wet ingredients, combine, cook on griddle, serve with syrup.",
    "Recipe: Simple Tomato Soup. Ingredients: Canned tomatoes, onion, garlic, vegetable broth, basil. Steps: Sauté onion and garlic, add tomatoes and broth, simmer, blend, season with basil.",
    "Recipe: Quick Chicken Stir-fry. Ingredients: Chicken breast, bell peppers, broccoli, soy sauce, ginger, garlic. Steps: Slice chicken and veggies, stir-fry chicken, add veggies, add sauce ingredients, cook until tender-crisp.",
    "Recipe: Basic Guacamole. Ingredients: Avocados, red onion, cilantro, lime juice, salt. Steps: Mash avocados, finely chop onion and cilantro, mix all ingredients, season to taste.",
    "Recipe: Chocolate Chip Cookies. Ingredients: Butter, sugar, brown sugar, eggs, flour, chocolate chips, baking soda. Steps: Cream butter and sugars, beat in eggs, mix in dry ingredients, fold in chips, bake until golden.",
    "Instruction: How to Change a Lightbulb. Steps: Turn off power at the switch, unscrew old bulb carefully, screw in new bulb clockwise until snug, turn power back on to test.",
    "Instruction: How to Restart Your Router. Steps: Locate the power button or unplug from wall, wait 10-15 seconds, plug back in or press power button, wait 1-2 minutes for lights to stabilize.",
    "Instruction: How to Set an Alarm on a Smartphone. Steps: Open 'Clock' app, tap 'Alarm' tab, tap '+' icon, set desired time, choose repeat days, save.",
    "Instruction: How to Brew Coffee with a Drip Machine. Steps: Fill water reservoir, add coffee grounds to filter basket, place carafe, press 'Brew' button.",
    "Instruction: How to Fasten a Seatbelt. Steps: Pull the metal tongue across your body, insert into the buckle until it clicks, pull strap to tighten, press button on buckle to release."
]

print(f"Generated {len(junk_texts)} junk texts.")

**Reasoning**:
Now that the 'junk' texts are generated, I will convert them into `Document` objects, assign unique IDs, and add them to the vector store with a specific metadata tag to identify them as 'junk'.



In [ ]:
junk_docs = []
junk_ids = []

# Start IDs from the current total number of documents to ensure uniqueness
start_id_for_junk = len(docs_with_director) + len(new_movie_entries)

for i, text in enumerate(junk_texts):
    metadata = {
        "type": "junk", # Identify these as junk documents
        "source_category": "recipe" if "Recipe:" in text else "instruction"
    }
    junk_docs.append(Document(page_content=text, metadata=metadata))
    junk_ids.append(str(start_id_for_junk + i))

print(f"Created {len(junk_docs)} new Document objects for junk texts.")
print(f"Generated {len(junk_ids)} unique IDs for junk documents.")

# Add the new documents to the vector store
vectorstore.add_documents(documents=junk_docs, ids=junk_ids)
print(f"Successfully added {len(junk_docs)} junk documents to the vector store.")


**Reasoning**:
To analyze if the 'junk' texts appear in the top-K results, I will perform abstract similarity searches using the `vectorstore` and inspect the returned documents.



In [ ]:
abstract_queries = [
    "Describe family dynamics in a dramatic setting.",
    "Instructions on how to perform a common household task.",
    "A quick guide for preparing a simple meal.",
    "Science fiction themes about humanity's future."
]

for query in abstract_queries:
    print(f"\n--- Query: '{query}' ---")
    results = vectorstore.similarity_search_with_score(query, k=5)

    found_junk = False
    for i, (doc, score) in enumerate(results, 1):
        similarity = round(1 - score, 4)
        doc_type = doc.metadata.get('type', 'N/A')
        doc_title = doc.metadata.get('title', 'N/A')
        doc_source_category = doc.metadata.get('source_category', 'N/A')

        print(f"  Rank {i}: Type: {doc_type}, Title/Content: {doc_title if doc_type != 'junk' else doc.page_content[:50] + '...'}, Similarity: {similarity:.4f}")
        if doc_type == 'junk':
            found_junk = True
            print(f"    (Junk document found! Category: {doc_source_category})")

    if not found_junk:
        print("  No junk documents found in top 5 results for this query.")

# Task 6
Record the initial number of documents in the vector store and measure the average search latency for a set of sample queries.

In [ ]:
print(f"Total documents in vector store: {vectorstore._collection.count()}")

sample_queries = abstract_queries # Reusing the abstract_queries for latency measurement

search_latencies = []
for query in sample_queries:
    start_time = time.time()
    vectorstore.similarity_search_with_score(query, k=5)
    end_time = time.time()
    search_latencies.append(end_time - start_time)

average_latency = sum(search_latencies) / len(search_latencies)
print(f"Average search latency for {len(sample_queries)} queries: {average_latency:.4f} seconds")

In [ ]:
def duplicate_documents(source_docs, start_id):
    duplicated_docs = []
    unique_ids = []
    for i, doc in enumerate(source_docs):
        # Create a new Document object with the same content and metadata
        new_doc = Document(page_content=doc.page_content, metadata=doc.metadata)
        # Generate a unique ID
        new_id = str(start_id + i)
        duplicated_docs.append(new_doc)
        unique_ids.append(new_id)
    return duplicated_docs, unique_ids

# Calculate start_id to ensure uniqueness across all existing documents
# len(docs_with_director) is the initial number of documents after adding director info
# len(new_movie_entries) is the number of new movies added
# len(junk_texts) is the number of junk texts added
current_total_docs = len(docs_with_director) + len(new_movie_entries) + len(junk_texts)

duplicated_docs, duplicated_ids = duplicate_documents(docs_with_director, current_total_docs)

print(f"Generated {len(duplicated_docs)} duplicated documents.")
print(f"Generated {len(duplicated_ids)} unique IDs for duplicated documents.")

In [ ]:
vectorstore.add_documents(documents=duplicated_docs, ids=duplicated_ids)
print(f"Successfully added {len(duplicated_docs)} duplicated documents to the vector store.")


**Reasoning**:
The error indicates that the batch size for adding documents exceeds ChromaDB's internal limit. To resolve this, I need to iterate through the `duplicated_docs` and `duplicated_ids` in smaller batches and add them to the vector store.



In [ ]:
batch_size = 2000 # Choose a batch size less than the max_batch_size (5461) disclosed in the error

for i in range(0, len(duplicated_docs), batch_size):
    batch_docs = duplicated_docs[i:i + batch_size]
    batch_ids = duplicated_ids[i:i + batch_size]
    vectorstore.add_documents(documents=batch_docs, ids=batch_ids)
    print(f"Successfully added batch {i // batch_size + 1} of {len(batch_docs)} documents to the vector store.")

print(f"Finished adding all {len(duplicated_docs)} duplicated documents to the vector store.")

In [ ]:
print(f"Total documents in vector store: {vectorstore._collection.count()}")

sample_queries = abstract_queries # Reusing the abstract_queries for latency measurement

search_latencies = []
for query in sample_queries:
    start_time = time.time()
    vectorstore.similarity_search_with_score(query, k=5)
    end_time = time.time()
    search_latencies.append(end_time - start_time)

average_latency = sum(search_latencies) / len(search_latencies)
print(f"Average search latency for {len(sample_queries)} queries: {average_latency:.4f} seconds")

In [ ]:
target_document_counts = [20000, 35000, 50000]

document_counts = []
latency_measurements = []

current_doc_count = vectorstore._collection.count()
print(f"Initial document count: {current_doc_count}")

# Re-using abstract_queries from Task 5 for latency measurement
# Ensure embeddings is available for latency measurement
if 'embeddings' not in locals():
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)

for target_count in target_document_counts:
    print(f"\n--- Reaching target count: {target_count} ---")
    docs_to_add_count = target_count - current_doc_count

    if docs_to_add_count <= 0:
        print(f"Current count ({current_doc_count}) already meets or exceeds target ({target_count}). Skipping.")
        # Still record current metrics if skipping, to have a complete record
        document_counts.append(current_doc_count)
        search_latencies = []
        for query in abstract_queries:
            start_time = time.time()
            vectorstore.similarity_search_with_score(query, k=5)
            end_time = time.time()
            search_latencies.append(end_time - start_time)
        average_latency = sum(search_latencies) / len(search_latencies)
        latency_measurements.append(average_latency)
        print(f"  Current doc count: {current_doc_count}, Average latency: {average_latency:.4f} seconds")
        continue

    additional_docs = []
    additional_ids = []
    num_generated_for_target = 0
    batch_size_for_generation = len(docs_with_director) # duplicate in chunks of original dataset size

    while num_generated_for_target < docs_to_add_count:
        # Ensure IDs are unique across all documents, starting from the current max ID
        next_start_id = vectorstore._collection.count() + len(additional_ids)

        # Generate a new set of duplicated documents from the original `docs_with_director`
        # Only take as many as needed to reach the target or a full batch
        num_to_take = min(docs_to_add_count - num_generated_for_target, batch_size_for_generation)

        if num_to_take <= 0: # Avoid generating empty batches if docs_to_add_count is small
            break

        temp_duplicated_docs, temp_duplicated_ids = duplicate_documents(docs_with_director[:num_to_take], next_start_id)

        additional_docs.extend(temp_duplicated_docs)
        additional_ids.extend(temp_duplicated_ids)
        num_generated_for_target += len(temp_duplicated_docs)

    print(f"  Generated {len(additional_docs)} documents to add for target {target_count}.")

    # Add generated documents to the vector store in smaller batches
    chroma_batch_size = 2000 # Keep this constant as per previous error resolution
    for i in range(0, len(additional_docs), chroma_batch_size):
        batch_d = additional_docs[i:i + chroma_batch_size]
        batch_id = additional_ids[i:i + chroma_batch_size]
        vectorstore.add_documents(documents=batch_d, ids=batch_id)
        print(f"    Added sub-batch {i // chroma_batch_size + 1}/{len(additional_docs) // chroma_batch_size + (1 if len(additional_docs) % chroma_batch_size > 0 else 0)} ({len(batch_d)} docs).")

    current_doc_count = vectorstore._collection.count()
    document_counts.append(current_doc_count)
    print(f"  Current total documents: {current_doc_count}")

    # Measure latency
    search_latencies = []
    for query in abstract_queries:
        start_time = time.time()
        vectorstore.similarity_search_with_score(query, k=5)
        end_time = time.time()
        search_latencies.append(end_time - start_time)
    average_latency = sum(search_latencies) / len(search_latencies)
    latency_measurements.append(average_latency)
    print(f"  Average search latency: {average_latency:.4f} seconds")

print("\n--- Final Results ---")
print(f"Document Counts at Stages: {document_counts}")
print(f"Average Latencies at Stages: {latency_measurements}")